# Улучшение SCE-Net: Transformer Attention и контекстная оценка пар

Этот ноутбук — **отдельный дизайн-док + заготовка кода** для развития базовой модели из `sce_net_fashion_compatibility.ipynb`.

Цели:
1. Предложить практичные варианты улучшения baseline SCE-Net.
2. Показать, как аккуратно добавить **transformer/attention блоки** без потери интерпретируемости similarity conditions.
3. Сделать модель, которая скорит пару `(item_a, item_b)` с учётом **контекста двух айтемов одновременно**, а не только независимых эмбеддингов.


## 0) Краткая стратегия улучшений

Ниже 4 уровня улучшений (от простого к более мощному):

- **L1. Сильнее objective + mining**: focal BCE/soft-margin triplet, hard negative mining, class-balanced sampling.
- **L2. Pairwise head**: поверх эмбеддингов добавить MLP-главу на признаках `[a, b, |a-b|, a*b]`.
- **L3. Cross-attention**: контекстуализировать признаки `a` через `b` и наоборот (двунаправленно).
- **L4. Set/Outfit context**: учитывать не только пару, но и mini-context (категория, сезон, стиль, соседние айтемы).

На практике рекомендую стартовать с **L2 + L3**: это даёт резкий прирост качества при умеренной сложности.


## 1) Импорты и конфиг экспериментов


In [ ]:
import math
from dataclasses import dataclass
from typing import Dict, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class ExpConfig:
    d_model: int = 512
    n_heads: int = 8
    ff_mult: int = 4
    n_layers: int = 2
    dropout: float = 0.1
    temperature: float = 0.07
    pair_loss_weight: float = 1.0
    metric_loss_weight: float = 0.3


## 2) Базовые строительные блоки

Используем precomputed image-embedding (например, FashionCLIP из baseline), затем адаптер + transformer encoder.


In [ ]:
class MLPAdapter(nn.Module):
    """Лёгкий адаптер для перехода от d_in к d_model."""
    def __init__(self, d_in: int, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(d_in),
            nn.Linear(d_in, d_model),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, ff_mult: int = 4, dropout: float = 0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=n_heads,
            dropout=dropout,
            batch_first=True,
        )
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_model * ff_mult),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * ff_mult, d_model),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-LN
        h = self.ln1(x)
        attn_out, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.drop(attn_out)

        h = self.ln2(x)
        x = x + self.drop(self.ff(h))
        return x


## 3) Pair Context Transformer

Идея: формируем токены пары `[CLS, item_a, item_b, delta, prod]` и прогоняем через несколько transformer-блоков.
Это позволяет модели:
- выделять важные взаимодействия между айтемами,
- учитывать асимметрии (например, обувь к платью vs платье к обуви),
- строить более богатый pair-level сигнал.


In [ ]:
class PairContextTransformer(nn.Module):
    def __init__(self, d_in: int, cfg: ExpConfig):
        super().__init__()
        self.cfg = cfg
        self.adapter = MLPAdapter(d_in, cfg.d_model, cfg.dropout)

        self.cls = nn.Parameter(torch.zeros(1, 1, cfg.d_model))
        self.pos = nn.Parameter(torch.randn(1, 5, cfg.d_model) * 0.02)  # CLS, a, b, |a-b|, a*b

        self.blocks = nn.ModuleList([
            TransformerBlock(cfg.d_model, cfg.n_heads, cfg.ff_mult, cfg.dropout)
            for _ in range(cfg.n_layers)
        ])
        self.final_ln = nn.LayerNorm(cfg.d_model)

        # Pair classification head
        self.pair_head = nn.Sequential(
            nn.Linear(cfg.d_model, cfg.d_model),
            nn.GELU(),
            nn.Dropout(cfg.dropout),
            nn.Linear(cfg.d_model, 1),
        )

        # Optional metric projection for retrieval-like objectives
        self.metric_proj = nn.Linear(cfg.d_model, cfg.d_model)

    def forward(self, emb_a: torch.Tensor, emb_b: torch.Tensor) -> Dict[str, torch.Tensor]:
        # emb_a/emb_b: [B, d_in]
        a = self.adapter(emb_a)
        b = self.adapter(emb_b)
        delta = torch.abs(a - b)
        prod = a * b

        B = a.shape[0]
        cls = self.cls.expand(B, -1, -1)
        tokens = torch.stack([a, b, delta, prod], dim=1)  # [B,4,D]
        x = torch.cat([cls, tokens], dim=1) + self.pos

        for blk in self.blocks:
            x = blk(x)
        x = self.final_ln(x)

        cls_out = x[:, 0]
        logit = self.pair_head(cls_out).squeeze(-1)
        metric = F.normalize(self.metric_proj(cls_out), dim=-1)

        return {
            "logit": logit,
            "pair_repr": cls_out,
            "metric_repr": metric,
            "tokens": x,
        }


## 4) Лоссы: классификация + metric component

Комбинируем:
- `BCEWithLogitsLoss` (основной сигнал совместимости),
- `SupCon / InfoNCE` или triplet на `metric_repr` для более устойчивой геометрии.


In [ ]:
def pair_bce_loss(logit: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    y = y.float()
    return F.binary_cross_entropy_with_logits(logit, y)


def contrastive_nce(z: torch.Tensor, y: torch.Tensor, temperature: float = 0.07) -> torch.Tensor:
    """
    Простой supervised contrastive NCE на уровне пар.
    Позитивы: пары с одинаковым y (можно усложнить и делать позитивы только среди good).
    """
    z = F.normalize(z, dim=-1)
    sim = z @ z.T / temperature

    y = y.view(-1, 1)
    pos_mask = (y == y.T).float()
    eye = torch.eye(z.size(0), device=z.device)
    pos_mask = pos_mask - eye

    logits = sim - 1e9 * eye
    log_prob = logits - torch.logsumexp(logits, dim=1, keepdim=True)

    pos_count = pos_mask.sum(dim=1).clamp_min(1.0)
    loss = -(pos_mask * log_prob).sum(dim=1) / pos_count
    return loss.mean()


def total_loss(output: Dict[str, torch.Tensor], y: torch.Tensor, cfg: ExpConfig) -> Dict[str, torch.Tensor]:
    bce = pair_bce_loss(output["logit"], y)
    nce = contrastive_nce(output["metric_repr"], y, cfg.temperature)
    total = cfg.pair_loss_weight * bce + cfg.metric_loss_weight * nce
    return {"loss": total, "bce": bce.detach(), "nce": nce.detach()}


## 5) Как внедрить в текущий baseline из `sce_net_fashion_compatibility.ipynb`

Практический план миграции:

1. **Оставить существующий pipeline данных** (CSV, split, dataloader).
2. В датасете добавить возврат `emb_a`, `emb_b` (или считать on-the-fly через encoder, если GPU позволяет).
3. Вместо чистого score из SCE-Net добавить новый модуль `PairContextTransformer`.
4. На валидации логировать: `ROC-AUC`, `PR-AUC`, `F1@best_threshold`, `Recall@K`.
5. Провести ablation:
   - baseline SCE-Net,
   - + pair head,
   - + 1 transformer layer,
   - + 2 layers + metric loss.

Важно: сначала фиксируйте энкодер изображений (frozen), потом partial unfreeze последних слоёв.


## 6) Вариант с явным cross-attention (item-to-item)

Если хотите моделировать именно взаимодействие `a<->b`, можно сделать 2 токена и отдельный блок cross-attention:

- `a' = CrossAttn(query=a, key=b, value=b)`
- `b' = CrossAttn(query=b, key=a, value=a)`
- затем классификатор по `[a', b', |a'-b'|, a'*b']`

Это обычно компактнее и иногда стабильнее при небольшом датасете.


In [ ]:
class BiCrossAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        self.ab = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ba = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ln_a = nn.LayerNorm(d_model)
        self.ln_b = nn.LayerNorm(d_model)

    def forward(self, a: torch.Tensor, b: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # a,b: [B,1,D]
        a2, _ = self.ab(self.ln_a(a), self.ln_b(b), self.ln_b(b), need_weights=False)
        b2, _ = self.ba(self.ln_b(b), self.ln_a(a), self.ln_a(a), need_weights=False)
        return a + a2, b + b2


## 7) Что даст лучший прирост в вашем кейсе (приоритет)

1. **Pairwise learning objective** (классификация пары + calibration threshold) — почти всегда must-have.
2. **Hard negatives** из той же категории/цветовой группы — сильный буст именно для fashion compatibility.
3. **Transformer/cross-attention** на pair tokens — буст в сложных стилевых сочетаниях.
4. **Meta-features** (категория, бренд, сезон, gender, occasion) как дополнительные токены.
5. **Two-stage training**: frozen encoder → частичный fine-tune.

Если данных мало (<100k пар), начните с 1-2 attention layers и сильной регуляризации (dropout, weight decay, early stopping).


## 8) Дорожная карта экспериментов (рекомендуемая)

- **Шаг A (1-2 дня):** baseline reproduction + pair MLP head.
- **Шаг B (2-3 дня):** PairContextTransformer (1 слой), подбор LR/WD.
- **Шаг C (2-3 дня):** 2 слоя + metric component + hard negatives.
- **Шаг D:** cross-attention variant + сравнение latency/quality.

Критерий успеха: прирост PR-AUC и стабильность Recall@K в retrieval-сценарии.
